# Preparacao dos dados SONDA para analise horaria

Este notebook consolida os arquivos `*_MD.dat` da pasta `2026/Dados SONDA`, que estao em resolucao de 10 minutos, e gera duas bases horarias:

- `sonda_horario_2024-05-01_a_2025-05-01.csv`
- `sonda_horario_2025-05-01_a_2026-02-01.csv`

As saidas sao salvas em `2026/dados_tratados_sonda`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)

cwd = Path.cwd()
PASTA_2026 = cwd if cwd.name == '2026' else cwd.parent / '2026'
PASTA_DADOS_BRUTOS_SONDA = PASTA_2026 / 'Dados SONDA'
PASTA_DADOS_TRATADOS_SONDA = PASTA_2026 / 'dados_tratados_sonda'
PASTA_DADOS_TRATADOS_SONDA.mkdir(parents=True, exist_ok=True)

PERIODOS = {
    '2024-05-01_a_2025-05-01': {
        'inicio': pd.Timestamp('2024-05-01 00:00:00'),
        'fim_exclusivo': pd.Timestamp('2025-05-01 00:00:00'),
        'arquivo_saida': 'sonda_horario_2024-05-01_a_2025-05-01.csv',
    },
    '2025-05-01_a_2026-02-01': {
        'inicio': pd.Timestamp('2025-05-01 00:00:00'),
        'fim_exclusivo': pd.Timestamp('2026-02-01 00:00:00'),
        'arquivo_saida': 'sonda_horario_2025-05-01_a_2026-02-01.csv',
    },
}

COLUNAS_MEDIA = ['tp_sfc', 'humid', 'press', 'ws10_avg']
COLUNA_DIRECAO = 'wd10_avg'
COLUNA_CHUVA = 'rain'
MARCADORES_INVALIDOS = [-9999, -999, 9999, 7999]

print('Pasta 2026:', PASTA_2026)
print('Dados brutos SONDA:', PASTA_DADOS_BRUTOS_SONDA)
print('Dados tratados SONDA:', PASTA_DADOS_TRATADOS_SONDA)

Pasta 2026: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026
Dados brutos SONDA: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\Dados SONDA
Dados tratados SONDA: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\dados_tratados_sonda


In [2]:
def parse_intervalo_arquivo(caminho: Path):
    padrao = r'CPA_(\d{4})_(\d{3})_a_(\d{3})_MD\.dat$'
    encontrado = re.search(padrao, caminho.name)
    if not encontrado:
        return None
    ano, doy_ini, doy_fim = encontrado.groups()
    ano = int(ano)
    doy_ini = int(doy_ini)
    doy_fim = int(doy_fim)
    inicio = pd.Timestamp(f'{ano}-01-01') + pd.Timedelta(days=doy_ini - 1)
    fim_exclusivo = pd.Timestamp(f'{ano}-01-01') + pd.Timedelta(days=doy_fim)
    return inicio, fim_exclusivo


def arquivo_intersecta_periodo(caminho: Path, inicio: pd.Timestamp, fim_exclusivo: pd.Timestamp):
    intervalo = parse_intervalo_arquivo(caminho)
    if intervalo is None:
        return False
    arq_inicio, arq_fim_exclusivo = intervalo
    return arq_inicio < fim_exclusivo and arq_fim_exclusivo > inicio


def selecionar_arquivos_md(inicio: pd.Timestamp, fim_exclusivo: pd.Timestamp):
    arquivos = []
    for ano in range(inicio.year, fim_exclusivo.year + 1):
        pasta_ano = PASTA_DADOS_BRUTOS_SONDA / str(ano)
        if not pasta_ano.exists():
            continue
        for caminho in sorted(pasta_ano.glob('*_MD.dat')):
            if arquivo_intersecta_periodo(caminho, inicio, fim_exclusivo):
                arquivos.append(caminho)
    return arquivos


for nome, cfg in PERIODOS.items():
    arquivos = selecionar_arquivos_md(cfg['inicio'], cfg['fim_exclusivo'])
    print('\n', nome)
    print('arquivos selecionados:', len(arquivos))
    for caminho in arquivos:
        print(' -', caminho.relative_to(PASTA_DADOS_BRUTOS_SONDA))


 2024-05-01_a_2025-05-01
arquivos selecionados: 12
 - 2024\CPA_2024_122_a_152_MD.dat
 - 2024\CPA_2024_153_a_182_MD.dat
 - 2024\CPA_2024_183_a_213_MD.dat
 - 2024\CPA_2024_214_a_244_MD.dat
 - 2024\CPA_2024_245_a_274_MD.dat
 - 2024\CPA_2024_275_a_305_MD.dat
 - 2024\CPA_2024_306_a_335_MD.dat
 - 2024\CPA_2024_336_a_366_MD.dat
 - 2025\CPA_2025_001_a_031_MD.dat
 - 2025\CPA_2025_032_a_059_MD.dat
 - 2025\CPA_2025_060_a_090_MD.dat
 - 2025\CPA_2025_091_a_120_MD.dat

 2025-05-01_a_2026-02-01
arquivos selecionados: 9
 - 2025\CPA_2025_121_a_151_MD.dat
 - 2025\CPA_2025_152_a_181_MD.dat
 - 2025\CPA_2025_182_a_212_MD.dat
 - 2025\CPA_2025_213_a_243_MD.dat
 - 2025\CPA_2025_244_a_273_MD.dat
 - 2025\CPA_2025_274_a_304_MD.dat
 - 2025\CPA_2025_305_a_334_MD.dat
 - 2025\CPA_2025_335_a_365_MD.dat
 - 2026\CPA_2026_001_a_031_MD.dat


In [3]:
def ler_arquivo_md(caminho: Path):
    # Linha 0: metadados TOA5; linha 1: cabecalho real; linhas 2 e 3: unidades/processamento.
    dados = pd.read_csv(caminho, skiprows=[0, 2, 3])
    dados['arquivo_origem'] = caminho.name
    dados['TIMESTAMP'] = pd.to_datetime(dados['TIMESTAMP'], errors='coerce')
    return dados


def media_circular_graus(valores):
    serie = pd.to_numeric(valores, errors='coerce').dropna()
    if serie.empty:
        return np.nan
    radianos = np.deg2rad(serie.to_numpy() % 360)
    seno_medio = np.sin(radianos).mean()
    cosseno_medio = np.cos(radianos).mean()
    if np.isclose(seno_medio, 0) and np.isclose(cosseno_medio, 0):
        return np.nan
    return float(np.rad2deg(np.arctan2(seno_medio, cosseno_medio)) % 360)


def timestamps_10min_ausentes(dados_10min, inicio, fim_exclusivo):
    esperados = pd.date_range(inicio, fim_exclusivo - pd.Timedelta(minutes=10), freq='10min')
    observados = pd.DatetimeIndex(dados_10min['TIMESTAMP'].dropna().sort_values().unique())
    return list(esperados.difference(observados))


def relatorio_lacunas_10min(dados_10min, inicio, fim_exclusivo):
    ausentes = timestamps_10min_ausentes(dados_10min, inicio, fim_exclusivo)
    if dados_10min.empty:
        return {'lacunas_10min': len(ausentes), 'maior_lacuna_min': np.nan, 'timestamps_ausentes': ausentes}
    datas = dados_10min['TIMESTAMP'].sort_values()
    diferencas = datas.diff().dropna()
    lacunas = diferencas[diferencas > pd.Timedelta(minutes=10)]
    return {
        'lacunas_10min': int(len(ausentes)),
        'maior_lacuna_min': float(lacunas.max().total_seconds() / 60) if len(lacunas) else 0.0,
        'timestamps_ausentes': ausentes,
    }


def localizar_arquivo_sd(timestamp):
    pasta_ano = PASTA_DADOS_BRUTOS_SONDA / str(timestamp.year)
    if not pasta_ano.exists():
        return None
    doy = int(timestamp.dayofyear)
    for caminho in sorted(pasta_ano.glob('*_SD.dat')):
        encontrado = re.search(r'_(\d{3})_a_(\d{3})_SD\.dat$', caminho.name)
        if not encontrado:
            continue
        inicio_doy, fim_doy = map(int, encontrado.groups())
        if inicio_doy <= doy <= fim_doy:
            return caminho
    return None


def ler_intervalo_sd(timestamp):
    caminho_sd = localizar_arquivo_sd(timestamp)
    if caminho_sd is None:
        return None, None
    inicio = timestamp
    fim_exclusivo = timestamp + pd.Timedelta(minutes=10)
    sd = pd.read_csv(caminho_sd, skiprows=[0, 2, 3])
    sd['TIMESTAMP'] = pd.to_datetime(sd['TIMESTAMP'], errors='coerce')
    sd = sd[(sd['TIMESTAMP'] >= inicio) & (sd['TIMESTAMP'] < fim_exclusivo)].copy()
    return caminho_sd, sd


def criar_registro_md_sintetico_por_sd(timestamp, colunas_md):
    colunas_necessarias = ['tp_sfc', 'humid', 'press', 'rain', 'ws10_avg', 'wd10_avg']
    caminho_sd, sd = ler_intervalo_sd(timestamp)
    info = {
        'timestamp': timestamp,
        'arquivo_sd': caminho_sd.name if caminho_sd is not None else None,
        'registros_sd': 0 if sd is None else len(sd),
        'status': 'nao_preenchido',
        'motivo': None,
    }
    if caminho_sd is None:
        info['motivo'] = 'arquivo SD nao encontrado'
        return None, info
    faltantes = [coluna for coluna in colunas_necessarias if coluna not in sd.columns]
    if faltantes:
        info['motivo'] = f'arquivo SD nao possui colunas meteorologicas necessarias: {faltantes}'
        return None, info
    if len(sd) == 0:
        info['motivo'] = 'nenhum registro SD no intervalo de 10 minutos'
        return None, info

    for coluna in colunas_necessarias:
        sd[coluna] = pd.to_numeric(sd[coluna], errors='coerce')
        sd.loc[sd[coluna].isin(MARCADORES_INVALIDOS), coluna] = np.nan

    registro = {coluna: np.nan for coluna in colunas_md}
    registro.update({
        'TIMESTAMP': timestamp,
        'RECORD': np.nan,
        'Id': np.nan,
        'Year': timestamp.year,
        'Jday': int(timestamp.dayofyear),
        'Min': timestamp.hour * 60 + timestamp.minute,
        'tp_sfc': sd['tp_sfc'].mean(),
        'humid': sd['humid'].mean(),
        'press': sd['press'].mean(),
        'rain': sd['rain'].sum(),
        'ws10_avg': sd['ws10_avg'].mean(),
        'wd10_avg': media_circular_graus(sd['wd10_avg']),
        'arquivo_origem': f'sintetico_por_SD:{caminho_sd.name}',
    })
    info['status'] = 'preenchido'
    info['motivo'] = 'registro sintetico criado a partir de SD'
    return pd.DataFrame([registro]), info


def preencher_lacunas_md_com_sd(dados, inicio, fim_exclusivo):
    lacunas_iniciais = relatorio_lacunas_10min(dados, inicio, fim_exclusivo)
    preenchimentos = []
    sinteticos = []
    for timestamp in lacunas_iniciais['timestamps_ausentes']:
        registro, info = criar_registro_md_sintetico_por_sd(timestamp, dados.columns)
        preenchimentos.append(info)
        if registro is not None:
            sinteticos.append(registro)
    if sinteticos:
        dados = pd.concat([dados, *sinteticos], ignore_index=True).sort_values('TIMESTAMP')
    return dados, lacunas_iniciais, preenchimentos


def consolidar_periodo(nome_periodo, cfg):
    inicio = cfg['inicio']
    fim_exclusivo = cfg['fim_exclusivo']
    arquivos = selecionar_arquivos_md(inicio, fim_exclusivo)
    if not arquivos:
        raise FileNotFoundError(f'Nenhum arquivo MD encontrado para {nome_periodo}')

    tabelas = [ler_arquivo_md(caminho) for caminho in arquivos]
    colunas_por_arquivo = {caminho.name: list(tabela.columns) for caminho, tabela in zip(arquivos, tabelas)}
    referencia = list(tabelas[0].columns)
    colunas_identicas = all(colunas == referencia for colunas in colunas_por_arquivo.values())

    dados = pd.concat(tabelas, ignore_index=True)
    dados = dados[(dados['TIMESTAMP'] >= inicio) & (dados['TIMESTAMP'] < fim_exclusivo)].copy()
    dados = dados.sort_values('TIMESTAMP')

    duplicadas = int(dados['TIMESTAMP'].duplicated().sum())
    dados = dados.drop_duplicates(subset='TIMESTAMP', keep='first')

    colunas_numericas = [coluna for coluna in dados.columns if coluna not in ['TIMESTAMP', 'arquivo_origem']]
    for coluna in colunas_numericas:
        dados[coluna] = pd.to_numeric(dados[coluna], errors='coerce')

    invalidos_por_coluna = {}
    for coluna in colunas_numericas:
        mascara = dados[coluna].isin(MARCADORES_INVALIDOS)
        invalidos_por_coluna[coluna] = int(mascara.sum())
        dados.loc[mascara, coluna] = np.nan

    dados, lacunas_iniciais, preenchimentos_sd = preencher_lacunas_md_com_sd(dados, inicio, fim_exclusivo)
    lacunas = relatorio_lacunas_10min(dados, inicio, fim_exclusivo)
    dados['data_hora'] = dados['TIMESTAMP'].dt.floor('h')

    agregacoes = {coluna: 'mean' for coluna in COLUNAS_MEDIA}
    agregacoes[COLUNA_CHUVA] = 'sum'
    horario = dados.groupby('data_hora').agg(agregacoes)
    horario[COLUNA_DIRECAO] = dados.groupby('data_hora')[COLUNA_DIRECAO].apply(media_circular_graus)
    horario['n_obs_hora'] = dados.groupby('data_hora')['TIMESTAMP'].count()
    horario['hora_incompleta'] = horario['n_obs_hora'] < 6
    horario = horario.reset_index()

    colunas_saida = ['data_hora', 'tp_sfc', 'humid', 'press', 'rain', 'ws10_avg', 'wd10_avg', 'n_obs_hora', 'hora_incompleta']
    horario = horario[colunas_saida]

    caminho_saida = PASTA_DADOS_TRATADOS_SONDA / cfg['arquivo_saida']
    horario.to_csv(caminho_saida, index=False)

    resumo = {
        'periodo': nome_periodo,
        'arquivo_saida': caminho_saida,
        'arquivos_lidos': len(arquivos),
        'colunas_identicas': colunas_identicas,
        'linhas_10min': len(dados),
        'inicio_10min': dados['TIMESTAMP'].min(),
        'fim_10min': dados['TIMESTAMP'].max(),
        'duplicadas_removidas': duplicadas,
        'lacunas_10min_iniciais': lacunas_iniciais['lacunas_10min'],
        'timestamps_ausentes_iniciais': lacunas_iniciais['timestamps_ausentes'],
        'preenchimentos_sd': preenchimentos_sd,
        'lacunas_10min': lacunas['lacunas_10min'],
        'timestamps_ausentes_finais': lacunas['timestamps_ausentes'],
        'maior_lacuna_min': lacunas['maior_lacuna_min'],
        'linhas_horarias': len(horario),
        'inicio_horario': horario['data_hora'].min(),
        'fim_horario': horario['data_hora'].max(),
        'horas_incompletas': int(horario['hora_incompleta'].sum()),
        'ausentes_horario': horario.isna().sum().to_dict(),
        'invalidos_10min': invalidos_por_coluna,
    }
    return horario, resumo


In [4]:
resultados = {}
resumos = []

for nome_periodo, cfg in PERIODOS.items():
    horario, resumo = consolidar_periodo(nome_periodo, cfg)
    resultados[nome_periodo] = horario
    resumos.append(resumo)
    print('\nPeriodo:', nome_periodo)
    print('Arquivo:', resumo['arquivo_saida'])
    print('Linhas 10 min:', resumo['linhas_10min'])
    print('Linhas horarias:', resumo['linhas_horarias'])
    print('Inicio/fim horario:', resumo['inicio_horario'], 'ate', resumo['fim_horario'])
    print('Horas incompletas:', resumo['horas_incompletas'])

pd.DataFrame([
    {
        'periodo': r['periodo'],
        'linhas_10min': r['linhas_10min'],
        'linhas_horarias': r['linhas_horarias'],
        'inicio_horario': r['inicio_horario'],
        'fim_horario': r['fim_horario'],
        'horas_incompletas': r['horas_incompletas'],
        'lacunas_10min_iniciais': r['lacunas_10min_iniciais'],
        'lacunas_10min_finais': r['lacunas_10min'],
        'maior_lacuna_min': r['maior_lacuna_min'],
    }
    for r in resumos
])


Periodo: 2024-05-01_a_2025-05-01
Arquivo: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\dados_tratados_sonda\sonda_horario_2024-05-01_a_2025-05-01.csv
Linhas 10 min: 52560
Linhas horarias: 8760
Inicio/fim horario: 2024-05-01 00:00:00 ate 2025-04-30 23:00:00
Horas incompletas: 0

Periodo: 2025-05-01_a_2026-02-01
Arquivo: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\dados_tratados_sonda\sonda_horario_2025-05-01_a_2026-02-01.csv
Linhas 10 min: 39742
Linhas horarias: 6624
Inicio/fim horario: 2025-05-01 00:00:00 ate 2026-01-31 23:00:00
Horas incompletas: 2


,periodo,linhas_10min,linhas_horarias,inicio_horario,fim_horario,horas_incompletas,lacunas_10min_iniciais,lacunas_10min_finais,maior_lacuna_min
0,2024-05-01_a_2025-05-01,52560,8760,2024-05-01,2025-04-30 23:00:00,0,0,0,0.0
1,2025-05-01_a_2026-02-01,39742,6624,2025-05-01,2026-01-31 23:00:00,2,2,2,20.0


In [5]:
linhas = []
linhas.append('Relatorio de qualidade - dados SONDA')
linhas.append('=' * 72)
linhas.append(f'Dados brutos: {PASTA_DADOS_BRUTOS_SONDA}')
linhas.append(f'Dados tratados: {PASTA_DADOS_TRATADOS_SONDA}')
linhas.append('')

for resumo in resumos:
    linhas.append(f"Periodo: {resumo['periodo']}")
    linhas.append('-' * 72)
    linhas.append(f"Arquivo de saida: {resumo['arquivo_saida']}")
    linhas.append(f"Arquivos MD lidos: {resumo['arquivos_lidos']}")
    linhas.append(f"Colunas identicas entre arquivos: {resumo['colunas_identicas']}")
    linhas.append(f"Linhas 10 min: {resumo['linhas_10min']}")
    linhas.append(f"Inicio/fim 10 min: {resumo['inicio_10min']} ate {resumo['fim_10min']}")
    linhas.append(f"Datas duplicadas removidas: {resumo['duplicadas_removidas']}")
    linhas.append(f"Lacunas 10 min iniciais: {resumo['lacunas_10min_iniciais']}")
    if resumo['timestamps_ausentes_iniciais']:
        linhas.append('Timestamps 10 min ausentes inicialmente:')
        for timestamp in resumo['timestamps_ausentes_iniciais']:
            linhas.append(f"  {timestamp}")
    linhas.append('Tentativas de preenchimento por SD:')
    if resumo['preenchimentos_sd']:
        for item in resumo['preenchimentos_sd']:
            linhas.append(
                f"  {item['timestamp']} | status={item['status']} | "
                f"arquivo_sd={item['arquivo_sd']} | registros_sd={item['registros_sd']} | "
                f"motivo={item['motivo']}"
            )
    else:
        linhas.append('  nenhuma lacuna detectada')
    linhas.append(f"Lacunas 10 min finais: {resumo['lacunas_10min']}")
    if resumo['timestamps_ausentes_finais']:
        linhas.append('Timestamps 10 min ainda ausentes:')
        for timestamp in resumo['timestamps_ausentes_finais']:
            linhas.append(f"  {timestamp}")
    linhas.append(f"Maior lacuna (min): {resumo['maior_lacuna_min']}")
    linhas.append(f"Linhas horarias: {resumo['linhas_horarias']}")
    linhas.append(f"Inicio/fim horario: {resumo['inicio_horario']} ate {resumo['fim_horario']}")
    linhas.append(f"Horas incompletas (<6 obs): {resumo['horas_incompletas']}")
    linhas.append('Ausentes no arquivo horario:')
    for coluna, total in resumo['ausentes_horario'].items():
        linhas.append(f"  {coluna}: {total}")
    linhas.append('Marcadores invalidos substituidos por NaN nos dados de 10 min:')
    for coluna, total in resumo['invalidos_10min'].items():
        if total:
            linhas.append(f"  {coluna}: {total}")
    linhas.append('')

caminho_relatorio = PASTA_DADOS_TRATADOS_SONDA / 'relatorio_qualidade_sonda.txt'
caminho_relatorio.write_text('\n'.join(linhas) + '\n', encoding='utf-8')
print('Relatorio salvo em:', caminho_relatorio)
print('\n'.join(linhas[:40]))

Relatorio salvo em: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\dados_tratados_sonda\relatorio_qualidade_sonda.txt
Relatorio de qualidade - dados SONDA
Dados brutos: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\Dados SONDA
Dados tratados: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\dados_tratados_sonda

Periodo: 2024-05-01_a_2025-05-01
------------------------------------------------------------------------
Arquivo de saida: c:\Users\sarah\OneDrive\Documentos\Sarah - FILES\Iniciação Cientifica - INPE\2026\dados_tratados_sonda\sonda_horario_2024-05-01_a_2025-05-01.csv
Arquivos MD lidos: 12
Colunas identicas entre arquivos: True
Linhas 10 min: 52560
Inicio/fim 10 min: 2024-05-01 00:00:00 ate 2025-04-30 23:50:00
Datas duplicadas removidas: 0
Lacunas 10 min iniciais: 0
Tentativas de preenchimento por SD:
  nenhuma lacuna detectada
Lacunas 10 min finais: 0
Maior lacuna (min): 0.0